# Expense Analysis

Load and inspect personal expense transactions from an Excel file.

In [20]:
import pandas as pd
from pathlib import Path

try:
    import openpyxl
except ImportError as exc:
    raise ImportError(
        "Missing dependency 'openpyxl'. Install it with `pip install openpyxl` or `pip install -r requirements.txt`."
    ) from exc

# Update this path if needed
excel_path = Path('/Users/brandontomlinson/Library/Mobile Documents/com~apple~CloudDocs/Excel/Copilot Model May 26.xlsx')

print('Loading:', excel_path)
if not excel_path.exists():
    raise FileNotFoundError(f"Excel file not found: {excel_path}")

xls = pd.ExcelFile(excel_path, engine='openpyxl')
print('Available sheets:', xls.sheet_names)

# Choose the sheet that contains your transaction detail
sheet_name = 'Data'  # correct sheet name
if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Using first sheet instead.")
    sheet_name = xls.sheet_names[0]
print('Using sheet:', sheet_name)

df = pd.read_excel(excel_path, sheet_name=sheet_name, engine='openpyxl')

print('Rows:', len(df))
print('Columns:', df.columns.tolist())
print(df.head())

Loading: /Users/brandontomlinson/Library/Mobile Documents/com~apple~CloudDocs/Excel/Copilot Model May 26.xlsx
Available sheets: ['BriModel', 'Bri Model', 'Combined Model', 'Detail1', 'Detail2', 'Detail3', 'Sheet5', 'Bri Data', 'Sheet1', 'Sheet6', 'Detail4', 'Detail5', 'Sheet1 (2)', 'Income Model', 'Sheet8', 'Sheet3', 'Data', 'Assets Model', 'FIRE', 'Assets Input', 'Sheet4', 'Sheet7', 'Historic Data', 'Fid Data', 'Sheet2', 'Cost per Meal', 'Cost per Mile', 'Exclusion Sheet']
Using sheet: Data
Rows: 11005
Columns: ['date', 'name', 'amount', 'status', 'category', 'parent category', 'excluded', 'tag', 'type', 'account', 'account mask', 'note', 'recurring', 'MnthYr', 'Exclude', 'Year', 'Exclude Current Month', 'Month']
        date                    name  amount   status       category  \
0 2024-08-04         Google Services    1.00  pending          Shops   
1 2024-08-04         Google Services   -1.00  pending          Shops   
2 2024-08-04         Google Services    2.11  pending       

In [21]:
# Basic inspection
if 'df' not in globals():
    raise RuntimeError("DataFrame 'df' is not defined. Run the first notebook cell until it finishes successfully.")

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11005 entries, 0 to 11004
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   11005 non-null  datetime64[ns]
 1   name                   11005 non-null  object        
 2   amount                 11005 non-null  float64       
 3   status                 11005 non-null  object        
 4   category               6898 non-null   object        
 5   parent category        2488 non-null   object        
 6   excluded               11005 non-null  bool          
 7   tag                    38 non-null     object        
 8   type                   11005 non-null  object        
 9   account                11005 non-null  object        
 10  account mask           7359 non-null   float64       
 11  note                   16 non-null     object        
 12  recurring              976 non-null    object        
 13  M

## Data preprocessing and overview

Parse dates, normalize column names, and identify key fields for category, vendor, amount, and date.

In [22]:
# Normalize column names and detect likely key fields
original_columns = df.columns.tolist()
df = df.copy()
df.columns = df.columns.str.strip()

candidates = {
    'date': [c for c in df.columns if 'date' in c.lower()],
    'category': [c for c in df.columns if 'cat' in c.lower() or 'category' in c.lower()],
    'vendor': [c for c in df.columns if 'vendor' in c.lower() or 'merchant' in c.lower() or 'payee' in c.lower()],
    'amount': [c for c in df.columns if 'amount' in c.lower() or 'amt' in c.lower() or 'value' in c.lower()]
}
print('Detected candidates:')
for key, vals in candidates.items():
    print(f"- {key}: {vals}")

# Adjust these names if your actual columns differ
date_col = candidates['date'][0] if candidates['date'] else None
category_col = candidates['category'][0] if candidates['category'] else None
vendor_col = candidates['vendor'][0] if candidates['vendor'] else None
amount_col = candidates['amount'][0] if candidates['amount'] else None

print('\nUsing columns:')
print('Date:', date_col)
print('Category:', category_col)
print('Vendor:', vendor_col)
print('Amount:', amount_col)

if date_col is not None:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    print('\nDate parse problems:', df[date_col].isna().sum(), 'missing')

print('\nMissing values by column:')
print(df.isna().sum())

if date_col is not None:
    df['Month'] = df[date_col].dt.to_period('M')

if amount_col is not None:
    df['Amount'] = pd.to_numeric(df[amount_col], errors='coerce')
    
# Keep only regular expense rows (rows where `type` == 'regular')
if 'type' in df.columns:
    prev_len = len(df)
    df = df[df['type'] == 'regular']
    print(f'Filtered to regular expenses: {len(df)} rows (removed {prev_len - len(df)})')

# Show a clean sample of the most relevant fields
sample_cols = [col for col in [date_col, category_col, vendor_col, amount_col, 'Month'] if col is not None]
df[sample_cols].head(10)

Detected candidates:
- date: ['date']
- category: ['category', 'parent category']
- vendor: []
- amount: ['amount']

Using columns:
Date: date
Category: category
Vendor: None
Amount: amount

Date parse problems: 0 missing

Missing values by column:
date                         0
name                         0
amount                       0
status                       0
category                  4107
parent category           8517
excluded                     0
tag                      10967
type                         0
account                      0
account mask              3646
note                     10989
recurring                10029
MnthYr                       0
Exclude                  10972
Year                         0
Exclude Current Month        0
Month                        0
dtype: int64
Filtered to regular expenses: 6902 rows (removed 4103)


,date,category,amount,Month
0,2024-08-04,Shops,1.00,2024-08
1,2024-08-04,Shops,-1.00,2024-08
2,2024-08-04,Shops,2.11,2024-08
3,2024-08-03,Restaurants,35.00,2024-08
4,2024-08-03,Subscriptions,15.49,2024-08
5,2024-08-03,Car,64.49,2024-08
8,2024-08-02,Groceries,88.00,2024-08
11,2024-08-02,Restaurants,33.02,2024-08
15,2024-08-01,Student Loan,174.77,2024-08
18,2024-08-01,Restaurants,28.25,2024-08


## Initial insights

Compute totals by category, top vendors, and monthly spend trends.

In [23]:
if amount_col is None:
    raise ValueError('No amount column detected. Update the notebook with your actual amount column name.')

total_spend = df['Amount'].sum()
print(f'Total spend loaded: ${total_spend:,.2f}')

if category_col is not None:
    category_summary = df.groupby(category_col)['Amount'].sum().sort_values(ascending=False)
    print('Top categories by spend:')
    display(category_summary.head(10))
    print('')
    print('Top 5 categories summary:')
    display(category_summary.head(5))
else:
    print('No category column detected. Please set `category_col` manually in the notebook.')

if vendor_col is not None:
    vendor_summary = df.groupby(vendor_col)['Amount'].sum().sort_values(ascending=False)
    print('Top vendors by spend:')
    display(vendor_summary.head(10))
else:
    print('No vendor column detected. Please set `vendor_col` manually in the notebook.')

if 'Month' in df.columns:
    monthly_summary = df.groupby('Month')['Amount'].sum().sort_index()
    monthly_summary_2026 = monthly_summary[monthly_summary.index.year == 2026]
    print('Monthly spend totals for 2026:')
    display(monthly_summary_2026)

    if category_col is not None:
        monthly_by_category = (
            df.groupby(['Month', category_col])['Amount']
            .sum()
            .unstack(fill_value=0)
            .sort_index()
        )
        monthly_by_category_2026 = monthly_by_category[monthly_by_category.index.year == 2026]
        print('Monthly expense by category for 2026:')
        display(monthly_by_category_2026)
    else:
        print('No category column detected, so monthly category breakdown cannot be computed.')
else:
    print('No month column available. Make sure the date column was parsed correctly.')


Total spend loaded: $375,463.65
Top categories by spend:


category
Rent             127255.09
Restaurants       42687.63
Car               34879.48
Home              22751.87
Travel            21703.98
Groceries         21527.45
Shops             21191.58
Entertainment     18125.67
Student Loan      10676.16
Subscriptions     10529.68
Name: Amount, dtype: float64


Top 5 categories summary:


category
Rent           127255.09
Restaurants     42687.63
Car             34879.48
Home            22751.87
Travel          21703.98
Name: Amount, dtype: float64

No vendor column detected. Please set `vendor_col` manually in the notebook.
Monthly spend totals for 2026:


Month
2026-01     8852.46
2026-02     7821.43
2026-03    13866.87
2026-04     9083.52
2026-05     8807.54
2026-06     4449.69
2026-08     2024.28
2026-10      244.00
2026-11      216.30
Freq: M, Name: Amount, dtype: float64

Monthly expense by category for 2026:


category,Bars & Nightlife,Car,Clothing,Donations,Entertainment,Groceries,Gym,Healthcare,Home,Other,Personal Care,Rent,Restaurants,Shops,Student Loan,Subscriptions,Transportation,Travel,Work Expenses
Month,,,,,,,,,,,,,,,,,,,
2026-01,266.42,846.35,96.43,0.0,703.25,505.69,123.41,0.00,420.86,99.00,0.0,2904.50,1112.58,421.81,119.18,226.31,183.87,822.80,0.0
2026-02,111.57,468.65,741.82,0.0,286.50,408.22,645.00,0.00,262.71,40.95,90.0,2907.34,665.25,487.43,119.18,228.31,94.50,264.00,0.0
2026-03,171.30,397.46,999.79,0.0,306.65,244.62,602.00,0.00,291.68,450.45,40.0,2911.92,809.05,249.30,5885.78,330.31,176.56,0.00,0.0
2026-04,227.26,466.75,252.25,0.0,1205.94,444.15,359.00,0.00,379.79,434.24,90.0,2912.41,1006.84,186.52,0.00,232.31,72.26,813.80,0.0
2026-05,339.78,505.80,190.29,100.0,339.35,284.58,876.24,0.00,239.24,131.96,35.0,2907.50,1374.16,85.71,0.00,423.28,193.51,781.14,0.0
2026-06,0.00,152.29,228.64,0.0,489.99,113.97,44.00,35.65,426.28,273.88,90.0,0.00,758.98,227.31,0.00,112.45,225.35,1270.90,0.0
2026-08,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,2024.28,0.0
2026-10,0.00,0.00,0.00,0.0,244.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
2026-11,0.00,0.00,0.00,0.0,216.30,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0


## Natural-language finance queries

Use `query_insights()` to ask questions like spending trends, top payees, forecasts, and categories.

In [27]:
from insights_helpers import query_insights

# Example natural-language queries
print('Trend:')
query_insights('how is my expense trending', df)

print('
Top spenders:')
query_insights('which name had the most expense', df, n=10)

print('
Forecast:')
query_insights('forecast next 6 months', df, periods=6)

print('
Categories:')
query_insights('show top categories', df)


SyntaxError: unterminated string literal (detected at line 7) (3153761418.py, line 7)